<a href="https://colab.research.google.com/github/techasit239/DADS7202_PigPicture/blob/main/FreshCheck_Phase1_Classification_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FreshCheck Deep Lab — Phase 1: Classification Baseline

**Step 2 · Experiment 1: EfficientNet-B0 fine-tune บน Kaggle Meat Freshness Dataset**

> 🔗 **ต้องรัน Phase 0 v3 (`FreshCheck_Phase0_Foundation_v3.ipynb`) ให้สำเร็จก่อน**  
> Phase 1 จะอ่าน splits CSV ที่ Phase 0 สร้าง → ถ้ายังไม่ได้รัน Phase 0 cell ที่ 1.1 จะ raise error ทันที

---

## Phase 1 ทำอะไร?

เทรน EfficientNet-B0 (pretrained บน ImageNet) ให้แยกเนื้อ Fresh / Half-Fresh / Spoiled  
ใช้เป็น **baseline** สำหรับเทียบกับ Swin-T (Exp 2) และ DINOv3 (Exp 3) ใน Phase 3

| ขั้น | ทำอะไร | เวลา |
|---|---|---|
| 1.0 | Setup environment | 30 วินาที |
| 1.1 | **โหลด Phase 0 splits + validate + copy data ไป /content** | 1-2 นาที |
| 1.2 | Dataset + Transforms + DataLoader | 10 วินาที |
| 1.3 | Visualize sample batch (sanity check) | 10 วินาที |
| 1.4 | Model setup (EfficientNet-B0) | 10 วินาที |
| 1.5 | Loss / Optimizer / Scheduler | 5 วินาที |
| 1.6 | **Training loop (15 epochs)** | **~15-20 นาที** |
| 1.7 | Final evaluation + confusion matrix | 30 วินาที |
| 1.8 | Save artifacts + summary | 5 วินาที |

**รวมทั้งหมด ~20-30 นาทีบน T4 GPU**

## เปิดใน Colab

1. `File → Upload notebook` → เลือกไฟล์นี้
2. `Runtime → Change runtime type → T4 GPU → Save`
3. `Runtime → Run all` (หรือรันทีละ cell)

## ลำดับการทำงานในทีม

```
   Phase 0 v3              Phase 1 v1 (ไฟล์นี้)         Phase 2 (อนาคต)
  ───────────────         ─────────────────────         ─────────────────
  download Kaggle    →   train EfficientNet-B0    →   3 segmentation exp
  group-aware split      บน Kaggle 80/20              (SAM 3 / Florence-2 / DINOv3)
  save splits/*.csv      save model + metrics
```


## 1.0 — Setup Environment

รวมทุก import + ตั้ง random seed + เช็ค GPU + mount Drive ในขั้นเดียว

**ทำไมต้องตั้ง seed?**  
Random seed ทำให้ผลลัพธ์เหมือนเดิมทุกครั้งที่รัน ถ้าไม่ตั้ง → train 2 ครั้งได้ accuracy ต่างกัน 1-2% เพราะ:
- การสุ่ม shuffle ข้อมูล
- การสุ่ม weight ตอน init head ใหม่
- การสุ่ม augmentation (random flip, crop)

ทุกการสุ่มใน Phase 1 ใช้ seed = 42 เพื่อให้ทีมเรียกผลซ้ำได้


In [ ]:
# 1.0.1 — Imports + seed + GPU check
import os, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T

from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
from tqdm.auto import tqdm

# Reproducibility — fix all random sources
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch:   {torch.__version__}')
print(f'CUDA:      {torch.cuda.is_available()}')
print(f'Device:    {device}')
if torch.cuda.is_available():
    print(f'GPU:       {torch.cuda.get_device_name(0)}')
    print(f'VRAM:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('[!] GPU ไม่พบ — Runtime → Change runtime type → T4 GPU')


In [ ]:
# 1.0.2 — Mount Drive + define paths
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/FreshCheck'

# Input (from Phase 0)
TRAIN_CSV = f'{PROJECT_ROOT}/splits/kaggle_train.csv'
VAL_CSV   = f'{PROJECT_ROOT}/splits/kaggle_val.csv'
DATA_DRIVE = f'{PROJECT_ROOT}/data/kaggle_meat'

# Output (Phase 1 will write here)
MODEL_DIR  = f'{PROJECT_ROOT}/models/classification'
LOGS_DIR   = f'{PROJECT_ROOT}/logs'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

BEST_MODEL_PATH = f'{MODEL_DIR}/phase1_efficientnet_b0_best.pth'
HISTORY_CSV     = f'{LOGS_DIR}/phase1_training_history.csv'
METRICS_JSON    = f'{LOGS_DIR}/phase1_metrics.json'
CM_PLOT_PATH    = f'{LOGS_DIR}/phase1_confusion_matrix.png'

# Local working copy (faster I/O than Drive)
DATA_LOCAL = '/content/data_local/kaggle_meat'

print('[OK] Drive mounted')
print(f'Inputs:  {TRAIN_CSV}')
print(f'         {VAL_CSV}')
print(f'Outputs: {BEST_MODEL_PATH}')


## 1.1 — Load Phase 0 Splits + Copy Data Locally

### ทำไมต้องคัดลอกข้อมูลมา /content?

Drive เร็วพอที่จะ "load โมเดล" แต่ **ช้าเกินไปสำหรับการ train**:
- อ่านไฟล์จาก Drive: ~30-60 มิลลิวินาที/รูป (FUSE overhead)
- อ่านจาก /content (local SSD): ~1-5 มิลลิวินาที/รูป

15 epochs × 1,812 รูป = **27,180 reads** — Drive จะใช้เวลาเพิ่มอีก 30-45 นาทีฟรีๆ

วิธีแก้: คัดลอกข้อมูลครั้งเดียวตอนต้น (~1-2 นาที) แล้ว train จาก local เร็วกว่ามาก

**ข้อเสีย:** /content reset เมื่อ Colab session หมด — ต้องคัดลอกใหม่ครั้งหน้า แต่ก็คุ้ม


In [ ]:
# 1.1.1 — Load CSVs + validate Phase 0 v3 contracts
#
# Phase 0 v3 ส่งออก splits CSV ที่มี contract ตามนี้:
#   - คอลัมน์: filename, path, class, kaggle_split, size_kb, piece_id
#   - คลาส: FRESH, HALF_FRESH, SPOILED (3 classes)
# ถ้า contract ใดผิด → raise error ทันที (fail loud ดีกว่า fail silent)

EXPECTED_COLS = {'filename', 'path', 'class', 'piece_id'}
EXPECTED_CLASSES = {'FRESH', 'HALF_FRESH', 'SPOILED'}

def _validate(df, name):
    missing = EXPECTED_COLS - set(df.columns)
    assert not missing, f'{name}: ขาดคอลัมน์ {missing} — Phase 0 v3 อาจไม่ตรงเวอร์ชัน'
    classes = set(df['class'].unique())
    bad = classes - EXPECTED_CLASSES
    assert not bad, f'{name}: เจอ class ที่ไม่คาดคิด {bad} (คาดหวัง {EXPECTED_CLASSES})'
    assert len(df) >= 100, f'{name}: มีแค่ {len(df)} rows — น้อยผิดปกติ'

assert os.path.exists(TRAIN_CSV), f'ไม่พบ {TRAIN_CSV} — รัน Phase 0 v3 ก่อน'
assert os.path.exists(VAL_CSV),   f'ไม่พบ {VAL_CSV} — รัน Phase 0 v3 ก่อน'
assert os.path.exists(DATA_DRIVE), f'ไม่พบ {DATA_DRIVE} — Phase 0 download ไม่สมบูรณ์'

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
_validate(train_df, 'train')
_validate(val_df,   'val')

print(f'[OK] Phase 0 v3 outputs validated')
print(f'\nTrain: {len(train_df)} rows | columns: {list(train_df.columns)}')
print(f'Val:   {len(val_df)} rows')
print(f'\nClass distribution (train):')
print(train_df['class'].value_counts().to_string())
print(f'\nClass distribution (val):')
print(val_df['class'].value_counts().to_string())


In [ ]:
# 1.1.2 — Copy data to local SSD for fast training
import shutil, time

if os.path.exists(DATA_LOCAL) and len(os.listdir(DATA_LOCAL)) > 0:
    print(f'[OK] ข้อมูล local มีอยู่แล้ว — ข้ามการคัดลอก')
else:
    print(f'กำลังคัดลอก {DATA_DRIVE} → {DATA_LOCAL}')
    print(f'(ใช้เวลา ~1-2 นาที สำหรับ 2,266 รูป)')
    t0 = time.time()
    shutil.copytree(DATA_DRIVE, DATA_LOCAL)
    print(f'[OK] เสร็จใน {time.time() - t0:.1f} วินาที')

# Rewrite path column → point to local copy
train_df['path'] = train_df['path'].str.replace(DATA_DRIVE, DATA_LOCAL, regex=False)
val_df['path']   = val_df['path'].str.replace(DATA_DRIVE, DATA_LOCAL, regex=False)

# Verify a sample path works
sample_path = train_df['path'].iloc[0]
assert os.path.exists(sample_path), f'Path remap ผิด: {sample_path}'
print(f'\n[OK] Path remap สำเร็จ — sample: {sample_path}')


## 1.2 — Dataset, Transforms, DataLoader

### 3 ส่วนทำหน้าที่ต่างกัน

| ส่วน | หน้าที่ |
|---|---|
| **Dataset** | บอกว่า "ข้อมูล 1 ตัวอย่างเป็นยังไง" (อ่านรูป + label) |
| **Transforms** | แปลงรูปก่อนป้อนโมเดล (resize, augment, normalize) |
| **DataLoader** | จัดการ batching + shuffle + parallel loading |

### Data augmentation สำหรับเนื้อสด — ต้องระวัง!

**สี = signal หลัก** — แดงสด = Fresh, น้ำตาลคล้ำ = Spoiled  
ถ้า augment สีมากเกินไป → ลบ signal ทิ้ง โมเดลเรียนรู้ผิด

**ใช้:**
- ✅ Random horizontal flip (เนื้อไม่มีทิศซ้าย-ขวา)
- ✅ Random resized crop (เพิ่มความหลากหลายของมุมมอง)
- ✅ Color jitter **เบาๆ** (brightness 15%, hue 2%)

**ไม่ใช้:**
- ❌ Vertical flip (ไม่ realistic — ใครจะถ่ายเนื้อกลับหัว)
- ❌ Heavy hue/saturation shift (ทำลาย signal สี)
- ❌ Grayscale (ลบ signal หลัก)

### ImageNet normalization

EfficientNet-B0 pretrain ด้วย ImageNet — ต้อง normalize ด้วย mean/std เดิม:
- mean = [0.485, 0.456, 0.406]
- std  = [0.229, 0.224, 0.225]

ไม่ทำตามนี้ = ใส่ข้อมูลผิด distribution → pretrained weights ไม่ทำงาน


In [ ]:
# 1.2.1 — Dataset class
LABEL_MAP = {'FRESH': 0, 'HALF_FRESH': 1, 'SPOILED': 2}
CLASSES   = ['FRESH', 'HALF_FRESH', 'SPOILED']  # index → name

class MeatFreshnessDataset(Dataset):
    """Read an image + return (tensor, label) pair."""
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = LABEL_MAP[row['class']]
        return img, label


# 1.2.2 — Transforms
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 224

train_tf = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    # Gentle color jitter — color is the signal
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_tf = T.Compose([
    T.Resize((256, 256)),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


# 1.2.3 — DataLoaders
BATCH_SIZE = 32     # T4 fits this comfortably for EfficientNet-B0
NUM_WORKERS = 2     # Colab gives 2 vCPUs

train_ds = MeatFreshnessDataset(train_df, transform=train_tf)
val_ds   = MeatFreshnessDataset(val_df,   transform=val_tf)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f'Train: {len(train_ds)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} samples, {len(val_loader)} batches')
print(f'Batch size: {BATCH_SIZE}')


## 1.3 — Inspect a Batch (Sanity Check)

**ทำไมต้อง visualize?**  
ก่อน train ต้องเช็คว่า:
1. รูปอ่านขึ้นได้จริง (ไม่ดำ ไม่เพี้ยน)
2. Augmentation ทำงาน (รูปแต่ละครั้งดูต่างกันเล็กน้อย)
3. Label ตรงกับรูป (โมเดลจะเรียน label นี้)

ถ้ารูปออกมาแปลก = ปัญหาใน Dataset / transforms → ต้องแก้ก่อน train  
ถ้าไม่เช็ค = train ไป 15 epochs แล้วเจอว่า input พัง = เสียเวลา


In [ ]:
# 1.3.1 — Show 8 sample images from training loader
def denormalize(tensor):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)


imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    img = denormalize(imgs[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(f'{CLASSES[labels[i]]}', fontsize=11)
    ax.axis('off')
plt.suptitle('Training batch (augmented + normalized → de-normalized for display)',
             fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nBatch tensor shape: {imgs.shape}  (B, C, H, W)')
print(f'Label tensor shape: {labels.shape}')
print(f'Pixel range:        [{imgs.min():.2f}, {imgs.max():.2f}]  (normalized, around 0)')


## 1.4 — Model Setup: EfficientNet-B0

### ทำไมเลือก EfficientNet-B0?

**1. Proposal commitment:** เป็น Step 2 Exp 1 ตามที่ submit ไป — ไม่เปลี่ยน

**2. Empirical evidence:** Bramantyo et al. [10] benchmark 5 backbones บน packaged-meat dataset → **EfficientNet-B0 ได้ accuracy สูงสุด 98.10%** สูงกว่า ViT-B/16 ที่ได้ 94.42%

**3. Practical:** B0 มี **5.3M parameters** เท่านั้น — train เร็ว memory น้อย เหมาะกับ Colab T4

### Fine-tuning strategy

```
Original EfficientNet-B0:
  [Feature Extractor (5.3M params, ImageNet pretrained)] → [Classifier: 1280 → 1000 classes]
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^                ^^^^^^^^^^^^^^^^^^^^^^^^
                       เรียนรู้แล้วเรื่อง edges/textures           output 1000 classes ของ ImageNet
                                                                  (ไม่ใช่สิ่งที่เราต้องการ)

Our modification:
  [Feature Extractor (frozen=NO, fine-tune=YES)] → [Classifier: 1280 → 3 classes]
                                                                      ^^^^^^^^^
                                                                      Fresh/HF/Spoiled
```

**Fine-tune ทั้งหมด** (ไม่ freeze) เพราะ:
- เนื้อสด ≠ ImageNet objects → features ระดับ low/mid อาจต้อง adapt
- Dataset ใหญ่พอ (1,812 รูป × augmentation) → ไม่ overfit ง่าย
- Dropout 0.3 ช่วย regularize


In [ ]:
# 1.4.1 — Load EfficientNet-B0 + replace classifier head
NUM_CLASSES = 3
DROPOUT = 0.3

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# Replace the classifier (default: Dropout(0.2) → Linear(1280, 1000))
num_features = model.classifier[1].in_features  # 1280
model.classifier = nn.Sequential(
    nn.Dropout(p=DROPOUT, inplace=True),
    nn.Linear(num_features, NUM_CLASSES),
)
model = model.to(device)

# Show param counts
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:>10,} ({total_params/1e6:.2f}M)')
print(f'Trainable parameters: {trainable_params:>10,} ({trainable_params/1e6:.2f}M)')
print(f'New classifier head:')
print(model.classifier)


## 1.5 — Loss, Optimizer, Scheduler

### 3 ตัวร่วมกันเป็นเครื่องเรียนรู้ของโมเดล

```
   logits = model(x)
              ↓
   loss = criterion(logits, y)      ← วัด "ผิดเท่าไหร่"
              ↓
   loss.backward()                  ← คำนวณ gradient
              ↓
   optimizer.step()                 ← อัพเดท weights
              ↓
   scheduler.step()                 ← ปรับ learning rate
```

### Class-weighted Cross Entropy — ทำไม?

จาก Phase 0: SPOILED มีแค่ 516 รูป (27%) — น้อยกว่า FRESH 672 รูป (37%)

ถ้าใช้ CE ปกติ → โมเดลเอนเอียงไปทาย FRESH (เพราะ FRESH เยอะกว่า)  
ใช้ class weights → "ผิด SPOILED แล้วโดน penalty หนักกว่า" → forces โมเดลเรียน SPOILED ดีขึ้น

**Formula (balanced):** `weight_i = total / (n_classes × count_i)`

| Class | Count | Weight |
|---|---|---|
| FRESH | 672 | 0.899 |
| HALF_FRESH | 624 | 0.968 |
| SPOILED | 516 | **1.171** ← penalty หนักสุด |

### AdamW + CosineAnnealingLR

- **AdamW** lr=1e-4: AdamW = Adam + decoupled weight decay. lr=1e-4 เป็น standard สำหรับ fine-tune pretrained model (ไม่เล็กไป ไม่ใหญ่ไป)
- **CosineAnnealingLR**: lr ค่อยๆ ลดเป็น cosine curve จาก 1e-4 → 1e-6 ใน 15 epochs ตอนแรก lr สูง = เรียนรู้เยอะ ตอนหลัง lr ต่ำ = ปรับละเอียด


In [ ]:
# 1.5.1 — Class weights from train distribution
class_counts = train_df['class'].value_counts()
total = class_counts.sum()
n_classes = len(CLASSES)
weights = torch.tensor(
    [total / (n_classes * class_counts[c]) for c in CLASSES],
    dtype=torch.float
).to(device)

print('Class weights:')
for cls, w in zip(CLASSES, weights):
    print(f'  {cls:12s}: {w.item():.4f}')


# 1.5.2 — Loss + Optimizer + Scheduler
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-2
EPOCHS = 15

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6,
)

print(f'\nLoss:      CrossEntropyLoss with class weights')
print(f'Optimizer: AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})')
print(f'Scheduler: CosineAnnealingLR ({LEARNING_RATE} → 1e-6 over {EPOCHS} epochs)')


## 1.6 — Training Loop

### โครงสร้างของ epoch หนึ่ง

```
For each epoch (1 to 15):
  ┌────────────────────────────┐
  │ TRAIN PHASE                │
  │  - model.train()           │  ← เปิด dropout + batch norm update
  │  - Loop batches in train   │
  │    - forward + loss        │
  │    - backward + step       │
  │  - Track train loss/acc    │
  └────────────────────────────┘
              ↓
  ┌────────────────────────────┐
  │ VAL PHASE (no gradient)    │
  │  - model.eval()            │  ← ปิด dropout + freeze BN
  │  - Loop batches in val     │
  │    - forward only          │
  │  - Track val loss/acc/F1   │
  └────────────────────────────┘
              ↓
  scheduler.step() — ลด lr
              ↓
  if val_f1 ดีกว่าเดิม → save model
```

### ทำไม save best ด้วย Macro-F1 ไม่ใช่ Accuracy?

**Accuracy** จะดูดีเกินจริงตอนข้อมูล imbalance  
สมมติทาย FRESH หมดเลย: accuracy = 37% (เพราะ FRESH 37% ของข้อมูล) → ดูเหมือนเรียนรู้บ้าง  
แต่ F1 ของ SPOILED = 0% (ไม่เคยทายถูก)  
→ **Macro-F1 = ค่าเฉลี่ย F1 ของทุก class** จับ class imbalance ได้ดีกว่า

### Early stopping (built-in)

ถ้า val Macro-F1 ไม่ดีขึ้น 5 epochs ติดกัน → หยุดเทรน ป้องกัน overfit + ประหยัดเวลา


In [ ]:
# 1.6.1 — Training + evaluation helpers

def train_one_epoch(model, loader, criterion, optimizer, device, epoch, total_epochs):
    """One pass over training data. Returns (avg_loss, accuracy)."""
    model.train()
    running_loss, running_correct, running_total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f'Epoch {epoch:2d}/{total_epochs} [train]', leave=False)
    for imgs, labels in pbar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss    += loss.item() * imgs.size(0)
        running_correct += (logits.argmax(1) == labels).sum().item()
        running_total   += imgs.size(0)
        pbar.set_postfix(loss=f'{loss.item():.3f}',
                         acc=f'{running_correct/running_total:.3f}')
    return running_loss / running_total, running_correct / running_total


@torch.no_grad()
def evaluate(model, loader, criterion, device, desc='val'):
    """One pass over validation. Returns (loss, accuracy, macro_f1, preds, labels)."""
    model.eval()
    running_loss, running_correct, running_total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in tqdm(loader, desc=desc, leave=False):
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(imgs)
        loss = criterion(logits, labels)

        running_loss    += loss.item() * imgs.size(0)
        preds = logits.argmax(1)
        running_correct += (preds == labels).sum().item()
        running_total   += imgs.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = running_correct / running_total
    f1  = f1_score(all_labels, all_preds, average='macro')
    return running_loss / running_total, acc, f1, all_preds, all_labels


In [ ]:
# 1.6.2 — Main training loop
PATIENCE = 5   # early stopping
history = []
best_f1 = 0.0
epochs_no_improve = 0

print(f'Starting training: {EPOCHS} epochs, {len(train_loader)} batches/epoch\n')

for epoch in range(1, EPOCHS + 1):
    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device, epoch, EPOCHS
    )

    # Validate
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, device, 'val')

    # LR step
    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()

    # Track + save best
    is_best = val_f1 > best_f1
    if is_best:
        best_f1 = val_f1
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    history.append({
        'epoch': epoch,
        'lr': current_lr,
        'train_loss': train_loss, 'train_acc': train_acc,
        'val_loss': val_loss,     'val_acc': val_acc,    'val_f1': val_f1,
    })

    marker = ' ★ best' if is_best else ''
    print(f'Epoch {epoch:2d}/{EPOCHS} | lr={current_lr:.1e} | '
          f'train: loss={train_loss:.3f} acc={train_acc:.3f} | '
          f'val: loss={val_loss:.3f} acc={val_acc:.3f} f1={val_f1:.3f}{marker}')

    if epochs_no_improve >= PATIENCE:
        print(f'\n[Early stopping] val F1 ไม่ดีขึ้น {PATIENCE} epochs ติดกัน — หยุดเทรน')
        break

print(f'\n[OK] Best val Macro-F1: {best_f1:.4f}')
print(f'     Saved to: {BEST_MODEL_PATH}')


In [ ]:
import json
import os
import pandas as pd

# กำหนด Path ของข้อมูล CVAT
CVAT_DIR = f'{PROJECT_ROOT}/data/cvat_test'
COCO_JSON = f'{CVAT_DIR}/annotations/instances_default.json'
IMAGES_DIR = f'{CVAT_DIR}/images'

def process_cvat_coco(json_path, img_dir):
    if not os.path.exists(json_path):
        print(f"[!] ไม่พบไฟล์ {json_path}")
        return None

    with open(json_path, 'r', encoding='utf-8') as f:
        coco = json.load(f)

    # Map Category ID -> Name
    # (สมมติว่าใน CVAT ตั้งชื่อ Label ไว้สอดคล้องกัน)
    cat_map = {cat['id']: cat['name'].upper().replace(' ', '_').replace('-', '_')
               for cat in coco['categories']}

    # Map Image ID -> Filename
    img_map = {img['id']: img['file_name'] for img in coco['images']}

    records = []
    for ann in coco['annotations']:
        img_id = ann['image_id']
        cat_id = ann['category_id']

        filename = img_map[img_id]
        class_name = cat_map[cat_id]

        # ปรับชื่อคลาสให้ตรงกับ Kaggle Dataset (เผื่อพิมพ์ไม่เหมือนกัน)
        if 'HALF' in class_name: class_name = 'HALF_FRESH'
        elif 'FRESH' in class_name: class_name = 'FRESH'
        elif 'SPOIL' in class_name: class_name = 'SPOILED'

        records.append({
            'filename': filename,
            'path': os.path.join(img_dir, filename),
            'class': class_name,
            'split': 'test_cvat'
        })

    df = pd.DataFrame(records)

    # กรณี 1 รูปมีการตี Bounding Box หลายอัน (หลาย Label)
    # สำหรับ Image Classification เราจะ drop duplicates ให้เหลือ 1 รูป = 1 label ด่วน
    df = df.drop_duplicates(subset=['filename']).reset_index(drop=True)
    return df

# สร้าง Test DataFrame
test_df = process_cvat_coco(COCO_JSON, IMAGES_DIR)

if test_df is not None:
    # Save ลง splits folder เหมือน train/val
    TEST_CSV = f'{PROJECT_ROOT}/splits/cvat_test.csv'
    test_df.to_csv(TEST_CSV, index=False)

    print(f"[OK] สร้าง Test Set จาก CVAT สำเร็จ: {len(test_df)} รูป")
    print(test_df['class'].value_counts().to_string())
    print(f"[OK] Saved to: {TEST_CSV}")

In [ ]:
# 1.6.3 — Plot training history
hist_df = pd.DataFrame(history)
hist_df.to_csv(HISTORY_CSV, index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], 'o-', label='train', color='#1e3a5f')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   'o-', label='val',   color='#c4583a')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(hist_df['epoch'], hist_df['train_acc'], 'o-', label='train', color='#1e3a5f')
axes[1].plot(hist_df['epoch'], hist_df['val_acc'],   'o-', label='val',   color='#c4583a')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].set_title('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Macro-F1 (val only)
axes[2].plot(hist_df['epoch'], hist_df['val_f1'], 'o-', color='#4a7c59')
axes[2].axhline(best_f1, linestyle='--', color='#4a7c59', alpha=0.4,
                label=f'best={best_f1:.3f}')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Macro-F1')
axes[2].set_title('Val Macro-F1'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{LOGS_DIR}/phase1_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'[OK] History saved: {HISTORY_CSV}')


## 1.7 — Final Evaluation on Best Model

### ทำไมต้องโหลด best กลับมา?

ตอน training loop จบ — model ที่อยู่ใน memory คือ epoch สุดท้าย (อาจ overfit)  
**best model = epoch ที่ val F1 สูงสุด** (saved บน disk)  
ต้องโหลดกลับมาเพื่อ report metrics ที่ถูกต้อง

### Confusion Matrix อ่านยังไง?

```
                 Predicted →
              FRESH  HF   SP
        ↓
   FRESH  [ 175    5    1 ]   ← 175 รูป Fresh ทายถูก, 5 รูปทายผิดเป็น HF, 1 รูปทายผิดเป็น SP
True HF   [  10  140   15 ]
   SP     [   2   12   94 ]
```

**Diagonal = ทายถูก** (อยากให้สูง)  
**Off-diagonal = ทายผิด** (อยากให้ต่ำ)

ดู pattern ได้ว่าโมเดล "สับสน" ระหว่าง class ไหนบ่อย — เช่นถ้า HF ↔ SP สับสนเยอะ = boundary ระหว่าง 2 class นี้ไม่ชัด อาจต้องปรับ labelling protocol


In [ ]:
# 1.7.1 — Load best model + run final eval
model.load_state_dict(torch.load(BEST_MODEL_PATH))
val_loss, val_acc, val_f1, all_preds, all_labels = evaluate(
    model, val_loader, criterion, device, 'final eval'
)

print(f'Final evaluation on best checkpoint:')
print(f'  Accuracy:    {val_acc:.4f}')
print(f'  Macro-F1:    {val_f1:.4f}')
print(f'  Loss:        {val_loss:.4f}\n')

print('Per-class report:')
print(classification_report(all_labels, all_preds,
                            target_names=CLASSES, digits=4))


In [ ]:
# 1.7.2 — Confusion matrix plot
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)   # row-normalized

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, mat, title, fmt in [
    (axes[0], cm,      'Confusion Matrix (counts)',      'd'),
    (axes[1], cm_norm, 'Confusion Matrix (normalized)',  '.2f'),
]:
    im = ax.imshow(mat, cmap='Blues')
    ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES)))
    ax.set_xticklabels(CLASSES, rotation=20); ax.set_yticklabels(CLASSES)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title)
    # Annotate cells
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            color = 'white' if mat[i, j] > mat.max() * 0.6 else 'black'
            ax.text(j, i, format(mat[i, j], fmt),
                    ha='center', va='center', color=color, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig(CM_PLOT_PATH, dpi=100, bbox_inches='tight')
plt.show()
print(f'[OK] Confusion matrix saved: {CM_PLOT_PATH}')


## 1.8 — Save Outputs + Summary

บันทึก final metrics ใน JSON เพื่อ Phase 3 อ่านไปเทียบกับ Exp 2 (Swin-T) + Exp 3 (DINOv3)


In [ ]:
# 1.8.1 — Save final metrics to JSON
per_class_f1 = f1_score(all_labels, all_preds, average=None)

metrics = {
    'experiment': 'Phase 1 — Step 2 Exp 1 (EfficientNet-B0)',
    'dataset': 'Kaggle Meat Freshness (pork-filtered, 80/20 group-aware split)',
    'config': {
        'model': 'efficientnet_b0',
        'pretrained': 'IMAGENET1K_V1',
        'image_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'epochs_run': len(history),
        'optimizer': 'AdamW',
        'lr': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'scheduler': 'CosineAnnealingLR',
        'dropout': DROPOUT,
        'class_weighted_ce': True,
        'seed': SEED,
    },
    'results': {
        'val_accuracy':     float(val_acc),
        'val_macro_f1':     float(val_f1),
        'val_loss':         float(val_loss),
        'per_class_f1': {
            cls: float(f1) for cls, f1 in zip(CLASSES, per_class_f1)
        },
        'best_epoch': int(hist_df.loc[hist_df['val_f1'].idxmax(), 'epoch']),
    },
    'output_files': {
        'best_model':   BEST_MODEL_PATH,
        'history_csv':  HISTORY_CSV,
        'metrics_json': METRICS_JSON,
        'cm_plot':      CM_PLOT_PATH,
    },
}

with open(METRICS_JSON, 'w') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f'[OK] Metrics saved: {METRICS_JSON}')


In [ ]:
# 1.8.2 — Final summary
print('=' * 60)
print('PHASE 1 — Step 2 Exp 1 (EfficientNet-B0) — DONE')
print('=' * 60)
print(f'\nResults on Kaggle validation set (n={len(val_df)}):')
print(f'  Accuracy:  {val_acc:.4f}')
print(f'  Macro-F1:  {val_f1:.4f}')
print(f'  Best epoch: {int(hist_df.loc[hist_df["val_f1"].idxmax(), "epoch"])}/{EPOCHS}')
print(f'\nPer-class Macro-F1:')
for cls, f1 in zip(CLASSES, per_class_f1):
    bar = '█' * int(f1 * 40)
    print(f'  {cls:12s}: {f1:.4f}  {bar}')

print(f'\nOutput files (in Google Drive):')
for label, path in metrics['output_files'].items():
    size = os.path.getsize(path) / 1024 if os.path.exists(path) else 0
    print(f'  {label:14s}: {path}  ({size:.1f} KB)')

print(f'\n[OK] Phase 1 complete — พร้อมไป Phase 2 (segmentation)')
